In [1]:
import socket
from collections.abc import Iterator

from pyspark.sql import SparkSession

spark: SparkSession = (
    SparkSession.builder
    .appName("NYC Taxi Cluster Check")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.executor.cores", "1")
    .config("spark.executor.memory", "1g")
    .config("spark.cores.max", "3")
    .config("spark.dynamicAllocation.enabled", "false")
    .getOrCreate()
)

def identify_worker(partition_id: int, values: Iterator[int]) -> Iterator[tuple[int, str, int]]:
    yield partition_id, socket.gethostname(), sum(values)

sc = spark.sparkContext
sc.setLogLevel("WARN")

results: list[tuple[int, str, int]] = (
    sc.parallelize(range(1, 31), 3)
    .barrier()
    .mapPartitionsWithIndex(identify_worker)
    .collect()
)

workers: set[str] = {host for _, host, _ in results}
total: int = sum(subtotal for _, _, subtotal in results)

print(f"Spark: {spark.version} | Master: {sc.master}")
for partition_id, host, subtotal in sorted(results):
    print(f"Partition {partition_id}: {host} | subtotal={subtotal}")
print(f"Workers: {sorted(workers)} | Total: {total}")

assert workers == {"spark-worker-1", "spark-worker-2", "spark-worker-3"}
assert total == 465

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/24 02:20:28 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark: 4.0.1 | Master: spark://spark-master:7077
Partition 0: spark-worker-1 | subtotal=55
Partition 1: spark-worker-3 | subtotal=155
Partition 2: spark-worker-2 | subtotal=255
Workers: ['spark-worker-1', 'spark-worker-2', 'spark-worker-3'] | Total: 465


In [2]:
spark.stop()